# 🧠 Système Multi-Agent D-A — Gemini (Colab Natif)

Système multi-agents autonome utilisant **Gemini AI** directement dans Colab (GRATUIT, sans clé API).

1. **4 agents spécialisés** : Générateur, Critique, Vérificateur, Gardien mémoire
2. **Exécution réelle** du code Python généré
3. **Boucle de correction** : score < 7 → révision
4. **Sauvegarde continue** sur Google Drive

⚠️ Exécuter les cellules dans l'ordre. Aucun compte externe requis.


In [ ]:
# === CELLULE 1: Installation ===
!pip install -q google-generativeai
import google.generativeai as genai
from google.colab import userdata, auth
import os, json, re, subprocess, textwrap, time
from datetime import datetime

# Gemini gratuit dans Colab
auth.authenticate_user()
genai.configure()

# Test
model = genai.GenerativeModel('gemini-2.0-flash')
r = model.generate_content('Dis bonjour en fran\u00e7ais, une phrase.')
print(f'\u2705 Gemini connect\u00e9: {r.text[:100]}')


In [ ]:
# === CELLULE 2: Monter Google Drive pour sauvegarde persistante ===
from google.colab import drive
import os

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/DA_MultiAgent'
os.makedirs(f'{SAVE_DIR}/rounds', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/emails', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/sessions', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/logs', exist_ok=True)
print(f'✅ Google Drive monté. Sauvegarde dans: {SAVE_DIR}')

In [ ]:
# === CELLULE 3: Agents Gemini specialises ===

EXIGENCES = '''EXIGENCES OBLIGATOIRES:
- Solutions PLUS PERFORMANTES que les meilleures existantes
- Resoudre des problemes ACTUELLEMENT NON RESOLUS
- Code Python EXECUTABLE (pas de pseudocode)
- EXECUTER le code et rapporter les VRAIS resultats
- Ratios >2x interessant, >5x publiable, >10x significatif
- Comparer avec l'etat de l'art (citer la reference)
- Verifier C = DA-AD non-trivial
- SAUVEGARDER tout
- Repondre en FRANCAIS'''

ROLES = {
    'generator': 'Tu es un mathematicien-chercheur. Tu generes des solutions ORIGINALES en theorie Discretisation-Approximation (D-A).\nLe commutateur C = DA - AD entre discretisation et approximation est ton outil principal.\nATTENTION: C peut etre trivial (matrice nulle si DA=I). Toujours verifier.\n' + EXIGENCES,

    'critic': 'Tu es un critique mathematique IMPITOYABLE. Tu verifies CHAQUE claim.\n- EXECUTE le code independamment\n- Verifie les FONDATIONS (C non-trivial? Borne theorique?)\n- Compare avec etat de art REEL\n- Score 0-10. Si < 7, REJETE.\n- Detecte les hallucinations (chiffres inventes, ratios inverses)\nErreur passee: confirmer que le code tourne sans verifier que les maths sont correctes.\n' + EXIGENCES,

    'verifier': 'Tu es un verificateur INDEPENDANT. Tu nas PAS acces aux critiques.\nTu recois uniquement le code et les resultats bruts.\n- Execute chaque code de zero\n- Verifie les fondations mathematiques\n- Cherche si la methode existe deja sous un autre nom\n' + EXIGENCES,

    'memory_guard': 'Tu es le gardien de memoire du systeme multi-agents.\n- Verifie que TOUS les rounds sont sauvegardes (pas de trous)\n- Maintiens un index complet\n- Detecte les incoherences (scores, fichiers manquants)\nReponds en FRANCAIS.'
}

class GeminiAgent:
    def __init__(self, role, system_prompt):
        self.role = role
        self.system_prompt = system_prompt
        self.model = genai.GenerativeModel(
            'gemini-2.0-flash',
            system_instruction=system_prompt
        )
        self.chat = self.model.start_chat(history=[])
        self.history = []

    def send(self, message, max_retries=3):
        for attempt in range(max_retries):
            try:
                response = self.chat.send_message(message)
                text = response.text
                self.history.append({'role': 'user', 'content': message[:200]})
                self.history.append({'role': 'agent', 'content': text[:200]})
                return text
            except Exception as e:
                if 'quota' in str(e).lower() or 'rate' in str(e).lower():
                    wait = 30 * (attempt + 1)
                    print(f'  Rate limit {self.role}, attente {wait}s...')
                    time.sleep(wait)
                else:
                    print(f'  Erreur {self.role}: {e}')
                    time.sleep(5)
        return f'ERREUR: {self.role} - max retries'

    def reset_chat(self):
        self.chat = self.model.start_chat(history=[])

def execute_code(code_str):
    '''Execute du code Python et retourne stdout+stderr.'''
    # Extraire le code des blocs markdown
    blocks = re.findall(r'```python
(.*?)```', code_str, re.DOTALL)
    if not blocks:
        blocks = re.findall(r'```
(.*?)```', code_str, re.DOTALL)
    if not blocks:
        return 'PAS DE CODE TROUVE'

    all_output = []
    for i, block in enumerate(blocks):
        try:
            result = subprocess.run(
                ['python3', '-c', block],
                capture_output=True, text=True, timeout=60
            )
            out = result.stdout[-2000:] if result.stdout else ''
            err = result.stderr[-1000:] if result.stderr else ''
            all_output.append(f'=== Bloc {i+1} ===
STDOUT:
{out}
STDERR:
{err}')
        except subprocess.TimeoutExpired:
            all_output.append(f'=== Bloc {i+1} === TIMEOUT (60s)')
        except Exception as e:
            all_output.append(f'=== Bloc {i+1} === ERREUR: {e}')

    return '
'.join(all_output)

# Creer les 4 agents
agents = {role: GeminiAgent(role, prompt) for role, prompt in ROLES.items()}
print(f'✅ 4 agents Gemini crees: {", ".join(agents.keys())}')


In [ ]:
# === CELLULE 4: Orchestrateur Multi-Agent Autonome ===

async def run_multi_agent_system():
    '''Boucle principale du systeme multi-agents Gemini.'''
    import asyncio

    state_file = f'{SAVE_DIR}/state.json'
    if os.path.exists(state_file):
        state = json.loads(open(state_file).read())
    else:
        state = {'round': 1, 'scores': [], 'corrections': 0}

    print(f'
🎯 Systeme multi-agents Gemini lance!')
    print(f'   Round de depart: {state["round"]}')
    print(f'   Scores precedents: {state["scores"][-5:]}')

    while True:
        n = state['round']
        ts = datetime.now().strftime('%H:%M:%S')
        print(f'
{"="*60}')
        print(f'ROUND {n} - {ts}')
        print(f'{"="*60}')

        # Contexte des rounds precedents
        history_ctx = ''
        for i in range(max(1, n-3), n):
            vf = f'{SAVE_DIR}/rounds/round_{i}.md'
            if os.path.exists(vf):
                history_ctx += open(vf).read()[:2000] + '
'

        # --- PHASE 1: Generation ---
        gen_prompt = f"Round {n}.\n\nHistorique recent:\n{history_ctx[:3000]}\n\nPropose UNE contribution D-A originale avec:\n1. Code Python complet executable (numpy/scipy)\n2. Resultats d execution reels attendus\n3. Comparaison etat de l art chiffree\n4. Ce qui est NOUVEAU (pas dans la litterature)\n\nIMPORTANT: Le commutateur C = DA - AD doit etre NON-TRIVIAL. Verifie que C != 0."

        print(f'  📐 Generateur...')
        gen_response = agents['generator'].send(gen_prompt)
        print(f'  ✓ Generateur: {len(gen_response)} chars')

        # Executer le code genere
        print(f'  ⚙️ Execution du code...')
        exec_result = execute_code(gen_response)
        print(f'  ✓ Execution: {len(exec_result)} chars output')

        # --- PHASE 2: Critique (boucle de correction) ---
        max_attempts = 3
        attempt = 0
        score = 0
        critic_response = ''

        while attempt < max_attempts and score < 7:
            attempt += 1
            critic_prompt = f"Verifie ce Round {n} (tentative {attempt}).\n\nREPONSE DU GENERATEUR:\n{gen_response[:6000]}\n\nRESULTATS EXECUTION REELLE:\n{exec_result[:3000]}\n\nScore de 0 a 10. Si < 7, explique PRECISEMENT quoi corriger.\nFormat: Score: X/10"

            print(f'  🔍 Critique (tentative {attempt})...')
            critic_response = agents['critic'].send(critic_prompt)
            score_match = re.search(r'[Ss]core\s*:?\s*(\d+(?:\.\d+)?)\s*/\s*10', critic_response)
            score = float(score_match.group(1)) if score_match else 0
            print(f'  📊 Score: {score}/10')

            if score < 7 and attempt < max_attempts:
                state['corrections'] += 1
                fix_prompt = f"Score {score}/10 - CORRIGE.\n\nCRITIQUE:\n{critic_response[:4000]}\n\nRESULTATS EXECUTION:\n{exec_result[:2000]}\n\nCorrige les erreurs et repropose. Score minimum: 7/10."

                print(f'  🔄 Correction...')
                gen_response = agents['generator'].send(fix_prompt)

                # Re-executer
                print(f'  ⚙️ Re-execution...')
                exec_result = execute_code(gen_response)

        # --- PHASE 3: Verificateur independant (tous les 5 rounds) ---
        verifier_response = ''
        if n % 5 == 0:
            verif_prompt = f"Verification independante round {n}.\n\nCode et resultats:\n{gen_response[:5000]}\n\nExecution:\n{exec_result[:3000]}\n\nVerifie: 1) Le code est correct 2) Les maths sont valides 3) C est non-trivial"

            print(f'  🔬 Verificateur independant...')
            verifier_response = agents['verifier'].send(verif_prompt)

        # --- PHASE 4: Gardien memoire ---
        mem_prompt = f"Round {n} termine. Score: {score}/10. Tentatives: {attempt}.\nHistorique scores: {state['scores'][-10:]}\nVerifie la coherence."
        agents['memory_guard'].send(mem_prompt)

        # --- SAUVEGARDE ---
        round_content = f"# Round {n} - Score: {score}/10 - Tentatives: {attempt}\nDate: {datetime.now().isoformat()}\n\n## Generateur\n{gen_response}\n\n## Resultats Execution\n{exec_result}\n\n## Critique\n{critic_response}\n\n## Verificateur Independant\n{verifier_response if verifier_response else 'N/A (tous les 5 rounds)'}\n"
        with open(f'{SAVE_DIR}/rounds/round_{n}.md', 'w') as f:
            f.write(round_content)

        state['scores'].append(score)
        state['round'] = n + 1
        with open(state_file, 'w') as f:
            json.dump(state, f, indent=2)

        print(f'  ✅ Round {n} COMPLET - Score: {score}/10')
        print(f'  📈 Scores: {state["scores"][-10:]}')
        print(f'  💾 Sauve: {SAVE_DIR}/rounds/round_{n}.md')

        # Reset chats tous les 10 rounds pour eviter context overflow
        if n % 10 == 0:
            for agent in agents.values():
                agent.reset_chat()
            print(f'  🔄 Chats reinitialises')

        # Pause anti rate-limit
        time.sleep(5)

print('✅ Orchestrateur defini. Executer la cellule suivante pour lancer.')


In [ ]:
# === CELLULE 5: LANCER LE SYSTEME ===
# Tourne en CONTINU jusqu'a interruption
import asyncio
import nest_asyncio
nest_asyncio.apply()
asyncio.get_event_loop().run_until_complete(run_multi_agent_system())


In [ ]:
# === CELLULE 6: Anti-deconnexion Colab ===
import IPython
display(IPython.display.Javascript('''
function keepAlive() {
    console.log("Keep alive: " + new Date().toISOString());
    document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(keepAlive, 60000);
console.log("Anti-deconnexion active");
'''))
print('\u2705 Anti-deconnexion Colab active')


In [ ]:
# === CELLULE 7: Rapport de statut ===
state_file = f'{SAVE_DIR}/state.json'
if os.path.exists(state_file):
    state = json.loads(open(state_file).read())
    print(f"Round actuel: {state['round']}")
    print(f"Scores: {state['scores']}")
    print(f"Corrections: {state.get('corrections', 0)}")
    if state['scores']:
        print(f"Score moyen: {sum(state['scores'])/len(state['scores']):.1f}/10")
    print(f"\nRounds sauvegardes:")
    import glob
    for f in sorted(glob.glob(f'{SAVE_DIR}/rounds/round_*.md')):
        size = os.path.getsize(f)
        print(f"  {os.path.basename(f)}: {size:,} bytes")
else:
    print('Systeme pas encore lance')
